## Import Libraries

In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

from collections import defaultdict
import random
import copy

from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, mean_absolute_error


In [ ]:
import torch
import torch.nn as nn 
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim 
from torch.optim.lr_scheduler import ReduceLROnPlateau


## Load Dataframe

In [ ]:
df_app = pd.read_csv('App_FeatureSelection.csv')
df_baseline = pd.read_csv('Baseline_FeatureSelection.csv')

In [ ]:
df_app = df_app.sort_values(by = ['user_id', 'date'])
df_app['date'] = pd.to_datetime(df_app['date'])
df_app['day_order'] = (df_app.date - df_app.groupby('user_order')['date'].transform('min')).dt.days + 1
df_app['date_diff'] = df_app.groupby('user_order')['date'].diff().dt.days

df_app = df_app.drop(index = [1124, 1125, 1126])
df_app = df_app[df_app['user_order'] != 35]
df_app = df_app.reset_index(drop = True)


In [ ]:
df_app.user_order.unique()

In [ ]:
df_app = df_app.groupby('user_order').head(27).reset_index(drop = True)

In [ ]:
df_app.groupby('user_order').size().values

## Data Preprocessing

### Baseline Data

In [ ]:
df_baseline

In [ ]:
df_baseline = df_baseline.drop(index = 34)

In [ ]:
df_baseline.columns

In [ ]:
df_baseline['user_order'] = df_baseline.index + 1

baseline_feature_list = ['자율신경 활성도', '자율신경 균형도', '심장 안정도', 'Tp', 'Apen', 'Srd',
       'Vlf(ln)', '심장 안정도 상태 - Encoded']

baseline_user_col = 'user_order'

In [ ]:
len(baseline_feature_list)

In [ ]:
df_baseline

In [ ]:
S = df_baseline[baseline_feature_list]

S_matrix = df_baseline[baseline_feature_list].to_numpy(dtype = np.float32)
S_user = df_baseline[baseline_user_col].to_numpy()

baseline_dict  = {u: S_matrix[i] for i, u in enumerate(S_user)}

In [ ]:
baseline_dict.keys()

### App Data

In [ ]:
df_app

In [ ]:
feature_list = ['phq1', 'phq2', 'meal_morning', 'meal_lunch', 'meal_dinner',
       'meal_night', 'sleep_satisfy', 'sleep_disturb', 'amount_beer_glass',
       'amount_soju_glass', 'sleep_minute_adjusted']

target_variable = 'tmrw_alcohol_desire'

In [ ]:
X = df_app[feature_list]
y = df_app[target_variable]

In [ ]:
X_train, y_train, S_train, user_train, day_train,  = [], [], [], [], []
X_val, y_val,  S_val, user_val, day_val = [], [], [], [], []
X_test, y_test, S_test, user_test, day_test = [], [], [], [], []

In [ ]:
window = 4
val_days = 3
test_days = 5

In [ ]:
df_app.user_order.unique()

In [ ]:
for user, user_data in df_app.groupby('user_order'):
    
    
    X_user = user_data[feature_list].to_numpy(dtype = np.float32)
    y_user = user_data[target_variable].to_numpy()
    
    T = len(user_data)
    
    # Index 기준: Starts from 0
    test_start_day = T - test_days
    val_start_day = T - (val_days + test_days)
    train_end_day = val_start_day - 1 
    
    S_user = baseline_dict[user]
    
    
    for start in range(0, T - window + 1):
        
        target_idx = start + window - 1
        
        X_window = X_user[start : start + window]
        y_window = y_user[target_idx] 
        y_window = int(y_window) - 1 # [1,2,3,4,5] -> [0,1,2,3,4]
        
        
        # 1. Check Within Window
        day_list = user_data['day_order'].iloc[start : start + window]
        day_array = np.array(day_list)
        flag = (np.diff(day_array) == 1).all()
        
        if flag != True:
            continue
        # ==================================================================
        
        # 2. Check Next Day Target
        if target_idx + 1 < T:
            last_input_day = user_data['day_order'].iloc[target_idx]
            next_day = user_data['day_order'].iloc[target_idx + 1]
        
            if next_day - last_input_day != 1:
                continue
        # ==================================================================
        
        # Train, Val, Test Split
        if target_idx <= train_end_day:
            X_train.append(X_window)
            y_train.append(y_window)
            user_train.append(user)
            S_train.append(S_user)
            day_train.append(target_idx)
            
        elif val_start_day <= target_idx < test_start_day:
            X_val.append(X_window)
            y_val.append(y_window)
            user_val.append(user)
            S_val.append(S_user)
            day_val.append(target_idx)
            
        elif test_start_day <= target_idx:
            X_test.append(X_window)
            y_test.append(y_window)
            user_test.append(user)
            S_test.append(S_user)
            day_test.append(target_idx)
        
        
        

## List -> numpy()

In [ ]:
X_train, y_train, user_train, day_train, S_train = np.array(X_train), np.array(y_train), np.array(user_train), np.array(day_train), np.array(S_train)
X_val, y_val, user_val, day_val, S_val = np.array(X_val), np.array(y_val), np.array(user_val), np.array(day_val), np.array(S_val)
X_test, y_test, user_test, day_test, S_test = np.array(X_test), np.array(y_test), np.array(user_test), np.array(day_test), np.array(S_test)

## numpy() -> pytorch Tensor

In [ ]:
X_train_tensor = torch.from_numpy(X_train)
y_train_tensor = torch.from_numpy(y_train).long()

X_val_tensor = torch.from_numpy(X_val)
y_val_tensor = torch.from_numpy(y_val).long()

X_test_tensor = torch.from_numpy(X_test)
y_test_tensor = torch.from_numpy(y_test).long()

user_train_tensor = torch.from_numpy(user_train)
user_val_tensor = torch.from_numpy(user_val)
user_test_tensor = torch.from_numpy(user_test)

S_train_tensor = torch.from_numpy(S_train)
S_val_tensor = torch.from_numpy(S_val)
S_test_tensor = torch.from_numpy(S_test)


day_train_tensor = torch.from_numpy(day_train)
day_val_tensor = torch.from_numpy(day_val)
day_test_tensor = torch.from_numpy(day_test)

## Define Tensor Dataset

In [ ]:
train_dataset = TensorDataset(X_train_tensor, S_train_tensor, y_train_tensor, user_train_tensor, day_train_tensor)
val_dataset = TensorDataset(X_val_tensor, S_val_tensor, y_val_tensor, user_val_tensor, day_val_tensor)
test_dataset = TensorDataset(X_test_tensor, S_test_tensor,y_test_tensor, user_test_tensor, day_test_tensor)

## DataLoader

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle= True, drop_last= False)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size = 256, shuffle=False, drop_last=False)

## Baseline Model

In [ ]:
class BaselinetoH0(nn.Module):
    
    def __init__(self, static_size, hidden_size, p_drop = 0.5):
        
        super().__init__() # Inheritizaiton
        
        self.BaselineModel = nn.Sequential(
            nn.Linear(static_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden_size, hidden_size)
        )
        
    def forward(self, s):
        # s: Baseline Data
        h0 = torch.tanh(self.BaselineModel(s))
        return h0

## Vanilla RNN Model

In [ ]:
class TinyVanillaRNN(nn.Module):
    
    def __init__(self, input_size, hidden_size, static_size, num_classes = 5, p_drop = 0.5):
        
        super().__init__() # Inheritance
        
        self.rnn = nn.RNN(
            input_size = input_size,
            hidden_size = hidden_size,
            num_layers = 1,
            batch_first = True, # To Match the shape of the data (Batch, Time Window, Feature)
            nonlinearity = 'tanh'
        )
        
        self.h0_encoder = BaselinetoH0(static_size = static_size, hidden_size= hidden_size, p_drop= p_drop)
        self.head = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x, s):
        # x : App Data -> (Batch, Window, Feature) 
        # s : Baseline Data ->  (Batch, static_size) 
        
        h0 = self.h0_encoder(s) # (Batch, Hidden Size)
        h0 = h0.unsqueeze(0) # (1, Batch, Hidden Size)
        
        out, h_n = self.rnn(x, h0) # h_n: (1, Batch, hidden_size)
        h_last = h_n[-1] # (Batch, hidden_size)
        
        y_hat = self.head(h_last) # (Batch, num_class)
        
        return y_hat

## Hyper Parameters

In [ ]:
window_size_list = [2,4,6,8]
hidden_size_list = [4,8,16,32]

In [ ]:
input_size = X_train_tensor.shape[2]
static_size = S_train.shape[1]
hidden_size = hidden_size_list[2]
EPOCHS = 70

## Set Seed

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

## Evaluation

### 1. Get Predicted Alcohol Desire Values

In [ ]:
@torch.no_grad()
def predict_on_loader(model, loader):
    
    model.eval() # Evaluation Mode
    rows = []
    
    for X_batch, S_batch, y_batch, user_batch, day_batch in loader:
        
        logits = model(X_batch, S_batch) # Predict Probabilities for Alcohol Desire
        pred = logits.argmax(dim = 1) # Get Actual Alcohol Desire
        
        rows.append(pd.DataFrame({
            "user_order": user_batch.detach().numpy(), # Tensor -> numpy
            "target_day": day_batch.detach().numpy(),
            "y_true": y_batch.detach().numpy(),
            "y_pred": pred.detach().numpy()
        }))
        
    
    return pd.concat(rows, ignore_index=True)

### 2. Get Evaluation Metrics

In [ ]:
@torch.no_grad()
def eval_metrics_on_loader(model, loader, criterion):
    
    model.eval() # Evaluation Mode
    
    
     # Intialize Dummy Values
    total = 0
    correct = 0
    loss_sum = 0.0

    # Initialize Lists for Summary
    all_pred = [] # Predicted Values
    all_true = [] # True Values
    all_user = [] # User Values
    all_day = [] # Day Information

    for X_batch, S_batch, y_batch, user_batch, day_batch in loader:
        
        logits = model(X_batch, S_batch)
        pred = logits.argmax(dim=1)

        loss = criterion(logits, y_batch)
        loss_sum += loss.item() * X_batch.size(0) # Get Loss Sum

        correct = correct + (pred == y_batch).sum().item()
        total = total + y_batch.size(0)

        # Pytorch Tensor -> Numpy() -> STAN Code의 Data Part와 대응됨
        all_pred.append(pred.detach().numpy())
        all_true.append(y_batch.detach().numpy())
        all_user.append(user_batch.detach().numpy())
        all_day.append(day_batch.detach().numpy())

    y_pred = np.concatenate(all_pred)
    y_true = np.concatenate(all_true)
    users = np.concatenate(all_user)
    days = np.concatenate(all_day)
    
    # Participant Wise Accuracy
    user_correct = defaultdict(int)
    user_total = defaultdict(int)
    
    for u, y_p, y_t in zip(users, y_pred, y_true):
        # Update User Total
        user_total[u] += 1
        
         # Update User Correct if actually correct
        if y_p == y_t:
            user_correct[u] += 1
            
    # Calculate User Accuracy
    user_acc = {u: user_correct[u] / user_total[u] for u in user_total} # u -  Key of the Dictionary

    metrics = {
        "loss": (loss_sum / total),
        "acc": correct / total,
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "mae": mean_absolute_error(y_true, y_pred),
        
        'user_acc' : user_acc,
    }
    
    return metrics

### 3. Train the Model Once

In [ ]:
def train_one_run(run_id, seed):
    
    # Set Seed
    set_seed(seed)

    # Define a New Model each Time
    model = TinyVanillaRNN(
    input_size=input_size,
    hidden_size=hidden_size,
    static_size=static_size,
    num_classes=5,
    p_drop=0.2
)
    
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    history = []
    
    best_val_loss = float('inf')
    best_epoch = None
    best_state = None
    

    # 2) train loop
    for epoch in range(EPOCHS):
        
        model.train() # Training Mode
        
        total_loss = 0.0
        train_correct = 0
        train_total = 0
        
        
        for X_batch, S_batch, y_batch, user_batch, day_batch in train_loader:


            optimizer.zero_grad() # Initialize Weights
            logits = model(X_batch, S_batch)
            loss = criterion(logits, y_batch) # Calculate Loss 
            loss.backward() # Back Propagation -> Calculate Derivatives
            optimizer.step() # Update Weights
            
            ttotal_loss = total_loss + loss.item() * X_batch.size(0)
            pred = logits.argmax(dim = 1)
            train_correct = train_correct + (pred == y_batch).sum().item()
            train_total = train_total + y_batch.size(0)
        
        train_metrics = eval_metrics_on_loader(model, train_loader, criterion=criterion)
        val_metrics   = eval_metrics_on_loader(model, val_loader,   criterion=criterion)

    # Append to the History List
        history.append({
                "run": run_id,
                "seed": seed,
                
                "epoch": epoch + 1,
                
                "train_loss": train_metrics["loss"],
                "train_acc": train_metrics["acc"],
                "train_macro_f1": train_metrics["macro_f1"],
                "train_weighted_f1": train_metrics["weighted_f1"],
                "train_mae": train_metrics["mae"],
                
                "val_loss": val_metrics["loss"],
                "val_acc": val_metrics["acc"],
                "val_macro_f1": val_metrics["macro_f1"],
                "val_weighted_f1": val_metrics["weighted_f1"],
                "val_mae": val_metrics["mae"],
            })
        
        # Update Best Checkpoint
        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            best_epoch = epoch + 1
            best_state = copy.deepcopy(model.state_dict())
        
    # Load Weights of the best checkpoint
    model.load_state_dict(best_state)  

    # After Total Model Training => Final Evaluation
    final_train_metrics = eval_metrics_on_loader(model, train_loader, criterion=criterion)
    final_val_metrics   = eval_metrics_on_loader(model, val_loader,   criterion=criterion)
    final_test_metrics = eval_metrics_on_loader(model, test_loader, criterion=criterion)
        

    summary = {
        "run": run_id,
        "seed": seed,
        
        # Add Best Epoch, Best Val_loss
        'best_epoch' : best_epoch,
        'best_val_loss': best_val_loss,

        # Train (final model)
        "train_loss": final_train_metrics["loss"],
        "train_acc": final_train_metrics["acc"],
        "train_macro_f1": final_train_metrics["macro_f1"],
        "train_weighted_f1": final_train_metrics["weighted_f1"],
        "train_macro_precision": final_train_metrics["macro_precision"],
        "train_macro_recall": final_train_metrics["macro_recall"],
        "train_mae": final_train_metrics["mae"],

        # Val (final model)
        "val_loss": final_val_metrics["loss"],
        "val_acc": final_val_metrics["acc"],
        "val_macro_f1": final_val_metrics["macro_f1"],
        "val_weighted_f1": final_val_metrics["weighted_f1"],
        "val_macro_precision": final_val_metrics["macro_precision"],
        "val_macro_recall": final_val_metrics["macro_recall"],
        "val_mae": final_val_metrics["mae"],

        # Test (final model)
        "test_loss": final_test_metrics["loss"],
        "test_acc": final_test_metrics["acc"],
        "test_macro_f1": final_test_metrics["macro_f1"],
        "test_weighted_f1": final_test_metrics["weighted_f1"],
        "test_macro_precision": final_test_metrics["macro_precision"],
        "test_macro_recall": final_test_metrics["macro_recall"],
        "test_mae": final_test_metrics["mae"],
        
        
    }
    
    # Participant Wise Accuracy
    train_user_acc = final_train_metrics['user_acc']
    val_user_acc = final_val_metrics['user_acc']
    test_user_acc = final_test_metrics['user_acc']
  
    
     # Get Predicted Alcohol Desire Values for each day
    df_test_pred = predict_on_loader(model, test_loader)
        
    return summary, history, test_user_acc, df_test_pred

## Initialize Seeds

In [ ]:
N_RUNS = 100
base_seed  = 2026
seeds = [base_seed + i for i in range(N_RUNS)]

## Initialize Summary Lists

In [ ]:
all_summaries = []
all_histories = []
all_user_accs = []
all_test_preds = []

In [ ]:
for id, seed in enumerate(seeds):
    summary, history, test_user_acc, df_test_pred = train_one_run(run_id=id, seed=seed)
    
    df_test_pred['run'] = id 
    df_test_pred['seed'] = seed
    
    all_summaries.append(summary)
    all_histories.extend(history)
    all_user_accs.append({"run": id, "seed": seed, **test_user_acc})
    all_test_preds.append(df_test_pred)

In [ ]:
# all_summaries

In [ ]:
# all_histories

In [ ]:
# all_test_preds

In [ ]:
# all_user_accs

In [ ]:
df_runs = pd.DataFrame(all_summaries)    
df_hist = pd.DataFrame(all_histories) 
df_user = pd.DataFrame(all_user_accs)
df_test_all = pd.concat(all_test_preds, ignore_index=True)

## Summary

In [ ]:
df_runs

In [ ]:
cols = ["train_acc", "train_macro_f1", "train_weighted_f1", "train_mae",
        "val_acc", "val_macro_f1", "val_weighted_f1", "val_mae",
        "test_acc", "test_macro_f1", "test_weighted_f1", "test_mae",]

In [ ]:
df_runs[cols].agg(["mean","std"]).T.round(4)

## User Wise Accuracy

In [ ]:
df_user

In [ ]:
user_cols = [c for c in df_user.columns if c not in ['run', 'seed']]

In [ ]:
df_user[user_cols].agg(['mean', 'std']).T

In [ ]:
df_user_summary = df_user[user_cols].agg(['mean', 'std']).T
df_user_summary.columns = ['acc_mean', 'acc_std']

In [ ]:
df_user_summary

In [ ]:
df_user_summary.round(4).to_csv('../4.Post Hoc/Best_df_user_MULTI.csv', index = True)

## Prediction of Alcohol Desire

In [ ]:
df_test_all

In [ ]:
df_test_all['target_day'] += 1
df_test_all['y_true'] += 1
df_test_all['y_pred'] += 1

In [ ]:
df_test_all

In [ ]:
df_test_all.groupby(['user_order', 'target_day'])['y_pred'].value_counts()

In [ ]:
df_test_all.groupby(['user_order', 'target_day'], as_index= False)['y_pred'].value_counts()

In [ ]:
# Check Tie
def check_mode_tie(x):
    value_counts = x.value_counts()
    top = value_counts.iloc[0]
    n_top = (value_counts ==  top).sum()
    
    return n_top > 1
    

In [ ]:
df_mode_tie = df_test_all.groupby(['user_order', 'target_day'])['y_pred'].apply(check_mode_tie)

In [ ]:
df_mode_tie.sum()

In [ ]:
df_mode = df_test_all.groupby(['user_order', 'target_day'], as_index= False).agg(
    y_true = ('y_true', 'first'),
    y_pred_mode = ('y_pred', lambda s: s.value_counts().idxmax()),
    y_pred_freq = ('y_pred', lambda s: s.value_counts().max()),
    n_runs = ('y_pred', 'size')
)

In [ ]:
df_mode

In [ ]:
for user in df_mode['user_order'].unique():
    df_user = df_mode[df_mode['user_order'] == user] 
    
    plt.figure(figsize = (9,5))
    
    plt.plot(df_user['target_day'], df_user['y_true'], marker = 'o', label = 'True')
    plt.plot(df_user['target_day '], df_user['y_pred_mode'], marker = 'o', alpha = 0.4, label = 'Predicted')
    
    plt.xlabel('Days')
    plt.ylabel('Alcohol Desire')
    
    plt.title(f'Participant {user}')
    
    plt.xticks(df_user['target_day'])
    plt.yticks(np.arange(5) + 1 )
    
    plt.grid()
    plt.legend()
    
    plt.tight_layout()
    plt.show()

## History

In [ ]:
df_hist

In [ ]:
hist_agg = df_hist.groupby('epoch', as_index= False).agg(
    train_loss_mean = ('train_loss', 'mean'),
    train_loss_std = ('train_loss', 'std'),
    val_loss_mean  =('val_loss','mean'),
    val_loss_std   =('val_loss','std'),

    train_acc_mean =('train_acc','mean'),
    train_acc_std  =('train_acc','std'),
    val_acc_mean   =('val_acc','mean'),
    val_acc_std    =('val_acc','std'),

)

In [ ]:
hist_agg

In [ ]:
best_median = df_runs['best_epoch'].median()
best_1IQR = df_runs['best_epoch'].quantile(0.25)
best_3IQR = df_runs['best_epoch'].quantile(0.75)

In [ ]:
plt.figure(figsize= (9,5))

plt.plot(hist_agg['epoch'], hist_agg['train_loss_mean'], label='Train Loss (mean)')
plt.plot(hist_agg['epoch'], hist_agg['val_loss_mean'],   label='Val Loss (mean)')


plt.fill_between(
    hist_agg['epoch'],
    hist_agg['train_loss_mean'] - hist_agg['train_loss_std'],
    hist_agg['train_loss_mean'] + hist_agg['train_loss_std'],
    alpha=0.2
)
plt.fill_between(
    hist_agg['epoch'],
    hist_agg['val_loss_mean'] - hist_agg['val_loss_std'],
    hist_agg['val_loss_mean'] + hist_agg['val_loss_std'],
    alpha=0.2
)

plt.axvline(best_median, linestyle = '--', label = f'Best Epoch (Median = {best_median:.0f})')
plt.axvspan(best_1IQR, best_3IQR, alpha = 0.1, label = f'IQR: [{best_1IQR:.0f}, {best_3IQR:.0f}]')

plt.title('Train vs Val Loss (avg over seeds)')
plt.xlabel('Epochs')
plt.ylabel('loss')
plt.grid(alpha=0.3)

plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(9,5))

plt.plot(hist_agg['epoch'], hist_agg['train_acc_mean'], label='Train Acc (mean)')
plt.plot(hist_agg['epoch'], hist_agg['val_acc_mean'],   label='Val Acc (mean)')

plt.fill_between(
    hist_agg['epoch'],
    hist_agg['train_acc_mean'] - hist_agg['train_acc_std'],
    hist_agg['train_acc_mean'] + hist_agg['train_acc_std'],
    alpha=0.2
)
plt.fill_between(
    hist_agg['epoch'],
    hist_agg['val_acc_mean'] - hist_agg['val_acc_std'],
    hist_agg['val_acc_mean'] + hist_agg['val_acc_std'],
    alpha=0.2
)

plt.axvline(best_median, linestyle = '--', label = f'Best Epoch (Median = {best_median:.0f})')
plt.axvspan(best_1IQR, best_3IQR, alpha = 0.1, label = f'IQR: [{best_1IQR:.0f}, {best_3IQR:.0f}]')

plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Train vs Val Accuracy (mean over seeds)')
plt.grid(alpha=0.3)

plt.legend()
plt.show()